In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
!pip install catboost
!pip install sentencepiece
!pip install sentencepiece protobuf
!pip install tiktoken sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00


In [3]:

!nvidia-smi

Wed Sep  2 06:09:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import gc
import json
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from catboost import CatBoostClassifier
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoImageProcessor,
    RobertaTokenizer,
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

# Определение устройства (GPU T4 / CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Используемое устройство:", DEVICE)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB\n")

Используемое устройство: cuda
GPU: Tesla T4 | VRAM: 14.56 GB



In [5]:
def extract_sentence_length_features(text):
    if not text or len(text.strip()) < 10:
        return 0.0, 0.0, 0.0

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if not sentences:
        return 0.0, 0.0, 0.0

    sentence_lengths = [len(s.split()) for s in sentences]
    avg_len = float(np.mean(sentence_lengths))
    len_var = float(np.var(sentence_lengths)) if len(sentence_lengths) > 1 else 0.0
    short_sentences = sum(1 for l in sentence_lengths if l < 4)
    short_ratio = float(short_sentences / len(sentences))

    return avg_len, len_var, short_ratio
class BinocularsFeatureExtractor:
    def __init__(self, observer_name="Qwen/Qwen2.5-1.5B", performer_name="Qwen/Qwen2.5-1.5B-Instruct", device=DEVICE):
        self.device = torch.device(device)
        print(f"Загрузка моделей Binoculars: {observer_name} & {performer_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(observer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        dtype = torch.float16 if self.device.type == "cuda" else torch.float32

        self.observer = AutoModelForCausalLM.from_pretrained(
            observer_name, torch_dtype=dtype, low_cpu_mem_usage=True
        ).to(self.device)

        self.performer = AutoModelForCausalLM.from_pretrained(
            performer_name, torch_dtype=dtype, low_cpu_mem_usage=True
        ).to(self.device)

        self.observer.eval()
        self.performer.eval()
        self.observer.config.use_cache = False
        self.performer.config.use_cache = False

        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
        print("✓ Текстовые модели Qwen 2.5 успешно загружены!\n")

    @torch.inference_mode()
    def calculate_score_from_ids(self, input_ids):
        attention_mask = torch.ones_like(input_ids)

        obs_output = self.observer(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        obs_logits = obs_output.logits[..., :-1, :].float()
        shift_labels = input_ids[..., 1:]

        obs_log_probs = torch.log_softmax(obs_logits, dim=-1)
        obs_gathered = torch.gather(obs_log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
        log_ppl = -obs_gathered.mean().item()
        del obs_output

        perf_output = self.performer(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        perf_logits = perf_output.logits[..., :-1, :].float()

        perf_log_probs = torch.log_softmax(perf_logits, dim=-1)
        perf_probs = torch.exp(perf_log_probs)
        cross_entropy = -(perf_probs * obs_log_probs).sum(dim=-1)
        log_x_ppl = cross_entropy.mean().item()

        del obs_logits, perf_logits, obs_log_probs, perf_log_probs, perf_probs, cross_entropy, perf_output, attention_mask

        return float(log_ppl / log_x_ppl if log_x_ppl != 0 else 1.0)

    @torch.inference_mode()
    def extract_window_features(self, text, window_size=50, step=15):
        if not text or len(text.strip()) < 10:
            return 1.0, 1.0, 0.0, ""

        try:
            inputs = self.tokenizer(text, return_tensors="pt", truncation=False)
            input_ids = inputs["input_ids"].to(self.device)
            total_tokens = input_ids.shape[1]

            if total_tokens <= window_size:
                score = self.calculate_score_from_ids(input_ids)
                decoded_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
                del input_ids
                return float(score), float(score), 0.0, decoded_text.strip()

            scores = []
            windows_tokens = []
            for start in range(0, total_tokens - window_size + 1, step):
                w_ids = input_ids[:, start:start + window_size]
                score = self.calculate_score_from_ids(w_ids)
                scores.append(score)
                windows_tokens.append(w_ids)

            del input_ids
            if not scores:
                return 1.0, 1.0, 0.0, ""

            min_idx = int(np.argmin(scores))
            most_ai_chunk = self.tokenizer.decode(windows_tokens[min_idx][0], skip_special_tokens=True).strip()

            scores = np.asarray(scores, dtype=np.float32)
            return float(np.min(scores)), float(np.mean(scores)), float(np.var(scores)), most_ai_chunk

        except Exception as e:
            print(f"Ошибка обработки текста: {e}")
            return 1.0, 1.0, 0.0, ""

# Инициализируем текстовый экстрактор
extractor = BinocularsFeatureExtractor(device=DEVICE)

Загрузка моделей Binoculars: Qwen/Qwen2.5-1.5B & Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Текстовые модели Qwen 2.5 успешно загружены!



In [6]:
class VisualOCREngine:
    def __init__(self, model_name="microsoft/trocr-base-printed", device=DEVICE):
        self.device = torch.device(device)
        print(f"Загрузка визуального трансформера {model_name}...")

        # Прямой импорт токенизатора без сбойных конвертеров
        image_processor = AutoImageProcessor.from_pretrained(model_name)
        tokenizer = RobertaTokenizer.from_pretrained(model_name)
        self.processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

        self.model = VisionEncoderDecoderModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        print("✓ Модуль Microsoft TrOCR успешно загружен на GPU!\n")

    def extract_text(self, image_path):
        """Оцифровка изображения в текстовую строку."""
        if not image_path or not os.path.exists(image_path):
            return ""
        try:
            image = Image.open(image_path).convert("RGB")
            pixel_values = self.processor(image, return_tensors="pt").pixel_values.to(self.device)

            with torch.no_grad():
                generated_ids = self.model.generate(pixel_values, max_new_tokens=1024)

            extracted_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            return extracted_text.strip()
        except Exception as e:
            print(f"[Предупреждение] Ошибка обработки изображения {image_path}: {e}")
            return ""

    def get_multimodal_features(self, image_path, text_extractor):
        """Оцифровывает картинку и рассчитывает скоры Binoculars для текста с картинки."""
        raw_image_text = self.extract_text(image_path)

        if len(raw_image_text) >= 10:
            print(f" -> TrOCR распознал текст: \"{raw_image_text[:60]}...\"")
            b_min, b_mean, b_var, _ = text_extractor.extract_window_features(raw_image_text)
            return {
                "has_image_text": 1,
                "ocr_bino_min": b_min,
                "ocr_bino_mean": b_mean,
                "ocr_bino_var": b_var,
                "extracted_image_text": raw_image_text
            }
        else:
            return {
                "has_image_text": 0,
                "ocr_bino_min": 1.0,
                "ocr_bino_mean": 1.0,
                "ocr_bino_var": 0.0,
                "extracted_image_text": ""
            }

# Инициализируем визуальный экстрактор
ocr_engine = VisualOCREngine(device=DEVICE)

Загрузка визуального трансформера microsoft/trocr-base-printed...


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Модуль Microsoft TrOCR успешно загружен на GPU!



In [7]:
def build_final_parquet(jsonl_input_path, parquet_output_path):
    if not os.path.exists(jsonl_input_path):
        print(f"[Ошибка] Файл {jsonl_input_path} не найден на левой панели!")
        return None

    records = []
    with open(jsonl_input_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    print(f"Загружено записей из JSONL: {len(records)}")
    ml_data = []
    processed_human_emails = set()

    for idx, row in enumerate(records):
        email_id = str(row.get('email_id'))
        var_idx = row.get('variant_index', 0)
        orig_body = str(row.get('original_body', ''))
        ai_body = str(row.get('ai_mutated_body', ''))

        # Человеческий текст считаем строго 1 раз на email_id (без дубликатов)
        if email_id not in processed_human_emails:
            o_min, o_mean, o_var, _ = extractor.extract_window_features(orig_body)
            o_avg_len, o_len_var, o_short_ratio = extract_sentence_length_features(orig_body)
            ml_data.append({
                'email_id': email_id, 'variant_index': -1, 'text_type': 'Human', 'is_ai': 0,
                'bino_min': o_min, 'bino_mean': o_mean, 'bino_var': o_var,
                'avg_sent_len': o_avg_len, 'sent_len_var': o_len_var, 'short_sent_ratio': o_short_ratio
            })
            processed_human_emails.add(email_id)

        # ИИ текст считаем для каждого варианта
        a_min, a_mean, a_var, _ = extractor.extract_window_features(ai_body)
        a_avg_len, a_len_var, a_short_ratio = extract_sentence_length_features(ai_body)
        ml_data.append({
            'email_id': email_id, 'variant_index': var_idx, 'text_type': 'AI', 'is_ai': 1,
            'bino_min': a_min, 'bino_mean': a_mean, 'bino_var': a_var,
            'avg_sent_len': a_avg_len, 'sent_len_var': a_len_var, 'short_sent_ratio': a_short_ratio
        })

        if idx % 10 == 0 and DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(ml_data)
    df.to_parquet(parquet_output_path, engine='pyarrow', index=False)
    print(f"[Успешно] Матрица признаков сохранена в: {parquet_output_path}")
    return df

DATASET_PATH = "mutated_spam_dataset.jsonl"
OUTPUT_PARQUET = "final_features.parquet"

df_features = build_final_parquet(DATASET_PATH, OUTPUT_PARQUET)

Загружено записей из JSONL: 40
[Успешно] Матрица признаков сохранена в: final_features.parquet


In [8]:
feature_cols = ['bino_min', 'bino_mean', 'bino_var', 'avg_sent_len', 'sent_len_var', 'short_sent_ratio']

def train_catboost_detector(df):
    if df is None:
        print("[Ошибка] Нет данных для обучения.")
        return None
    df_clean = df.dropna(subset=feature_cols + ['is_ai'])
    X = df_clean[feature_cols]
    y = df_clean['is_ai']

    model = CatBoostClassifier(
        iterations=150,
        depth=3,
        learning_rate=0.05,
        verbose=False,
        random_seed=42
    )
    model.fit(X, y)
    print("✓ Модель CatBoost успешно обучена на матрице признаков!\n")
    return model

cb_model = train_catboost_detector(df_features)

✓ Модель CatBoost успешно обучена на матрице признаков!



In [9]:
def extract_ai_generated_segments(text, window_size=50, step=15, confidence_threshold=75.0):
    if not text or len(text.strip()) < 10:
        return []

    try:
        inputs = extractor.tokenizer(text, return_tensors="pt").to(extractor.device)
        input_ids = inputs["input_ids"][0]
        total_tokens = len(input_ids)

        B_0 = 0.9015  # Порог Binoculars
        k = 35.0      # Коэффициент крутизны

        detected_windows = []

        def calc_confidence(score_val):
            return 1.0 / (1.0 + math.exp(-k * (B_0 - score_val))) * 100.0

        if total_tokens <= window_size:
            score = extractor.calculate_score_from_ids(input_ids.unsqueeze(0))
            conf = calc_confidence(score)
            if conf >= confidence_threshold:
                decoded = extractor.tokenizer.decode(input_ids, skip_special_tokens=True)
                detected_windows.append({"start": 0, "end": total_tokens, "text": decoded.strip(), "confidence": conf})
        else:
            for i in range(0, total_tokens - window_size + 1, step):
                w_ids = input_ids[i: i + window_size]
                score = extractor.calculate_score_from_ids(w_ids.unsqueeze(0))
                conf = calc_confidence(score)

                if conf >= confidence_threshold:
                    decoded = extractor.tokenizer.decode(w_ids, skip_special_tokens=True)
                    detected_windows.append({"start": i, "end": i + window_size, "text": decoded.strip(), "confidence": conf})

        if not detected_windows:
            return []

        detected_windows.sort(key=lambda x: x["start"])
        merged = [detected_windows[0]]

        for current in detected_windows[1:]:
            prev = merged[-1]
            if current["start"] <= prev["end"]:
                prev["end"] = max(prev["end"], current["end"])
                prev["confidence"] = max(prev["confidence"], current["confidence"])
                combined_tokens = input_ids[prev["start"]: prev["end"]]
                prev["text"] = extractor.tokenizer.decode(combined_tokens, skip_special_tokens=True).strip()
            else:
                merged.append(current)

        return merged

    except Exception as e:
        print(f"Ошибка локализации: {e}")
        return []

In [10]:
def analyze_email(raw_text, image_path=None):
    b_min, b_mean, b_var, _ = extractor.extract_window_features(raw_text)
    avg_len, len_var, short_ratio = extract_sentence_length_features(raw_text)

    X_infer = pd.DataFrame([{
        'bino_min': b_min, 'bino_mean': b_mean, 'bino_var': b_var,
        'avg_sent_len': avg_len, 'sent_len_var': len_var, 'short_sent_ratio': short_ratio
    }])

    prob_ai = cb_model.predict_proba(X_infer)[0][1] if cb_model else 0.0
    is_ai = int(cb_model.predict(X_infer)[0]) if cb_model else 0
    suspicious_segments = extract_ai_generated_segments(raw_text, confidence_threshold=70.0)

    # Мультимодальный анализ вложения (если прикреплена картинка)
    visual_report = None
    if image_path and os.path.exists(image_path):
        visual_report = ocr_engine.get_multimodal_features(image_path, extractor)

    return {
        "verdict": "AI Generated" if is_ai == 1 else "Human Written",
        "ai_probability": f"{prob_ai * 100:.2f}%",
        "features": {
            "bino_min": round(b_min, 4),
            "bino_var": round(b_var, 6),
            "avg_sent_len": round(avg_len, 2)
        },
        "visual_attachment": visual_report,
        "suspicious_segments_count": len(suspicious_segments),
        "segments": suspicious_segments
    }

In [17]:
test_image = "text.png"

print("=== ТЕСТ МУЛЬТИМОДАЛЬНОГО ИНФЕРЕНСА ===")
sample_text = "Dear user, we detected an unauthorized login to your account. Please check the attached screenshot."

result = analyze_email(sample_text, image_path=test_image)
print(json.dumps(result, indent=2, ensure_ascii=False))

=== ТЕСТ МУЛЬТИМОДАЛЬНОГО ИНФЕРЕНСА ===
{
  "verdict": "AI Generated",
  "ai_probability": "81.62%",
  "features": {
    "bino_min": 1.3676,
    "bino_var": 0.0,
    "avg_sent_len": 7.5
  },
  "visual_attachment": {
    "has_image_text": 0,
    "ocr_bino_min": 1.0,
    "ocr_bino_mean": 1.0,
    "ocr_bino_var": 0.0,
    "extracted_image_text": ""
  },
  "suspicious_segments_count": 0,
  "segments": []
}
